# Entrenamiento y evaluación de BETO

## Objetivo

Este cuaderno documenta el entrenamiento, la selección y la evaluación final de un modelo Transformer para la clasificación binaria de sentimiento en comentarios de estudiantes universitarios.

El procedimiento comprende:

1. configuración y verificación del entorno de ejecución;
2. validación de las particiones de datos;
3. entrenamiento de tres configuraciones de BETO, diferenciadas por la tasa de aprendizaje;
4. selección del mejor modelo utilizando exclusivamente el conjunto de validación;
5. conservación del checkpoint seleccionado;
6. evaluación final sobre el conjunto de prueba reservado.

Los experimentos comparados fueron:

| Experimento | Learning rate |
|---|---:|
| T1 | `1e-5` |
| T2 | `2e-5` |
| T3 | `3e-5` |

En los tres casos se utilizó BETO cased con `seed=42`, `max_length=128`, un máximo de 3 épocas, `weight_decay=0.01` y batch efectivo de 16. Durante el entrenamiento no se aplicaron pesos de clase, oversampling ni early stopping.

El conjunto de prueba se mantuvo separado durante el entrenamiento y la selección del modelo. Su uso predictivo se reservó exclusivamente para la evaluación final.

## Guía de reproducción

Este cuaderno puede utilizarse de dos maneras.

### Opción A — Reproducir el experimento completo

Esta opción vuelve a entrenar T1, T2 y T3 y realiza nuevamente la selección mediante el conjunto de validación.

**Requisitos:**

- Google Colab con una GPU CUDA disponible para el entrenamiento;
- acceso al repositorio del proyecto;
- `train.csv`;
- `validation.csv`;
- `test.csv`;
- `metadata.json`;
- `baseline_C1_metadata.json` para las verificaciones de la etapa final;
- acceso a Google Drive para conservar el checkpoint seleccionado.

El entrenamiento utiliza el commit:

`90afa082979bbe396f5a6a4906cadee9c351dd4f`

y las siguientes versiones de Hugging Face:

| Paquete | Versión |
|---|---:|
| `transformers` | `4.57.1` |
| `tokenizers` | `0.22.2` |
| `huggingface-hub` | `0.36.2` |
| `accelerate` | `1.10.1` |

El entrenamiento requiere GPU y FP16 porque esa fue la configuración experimental utilizada para T1, T2 y T3.

### Opción B — Reproducir únicamente la evaluación final

Esta opción no vuelve a entrenar. Requiere el checkpoint ya seleccionado:

`checkpoint-3036`

correspondiente al experimento T2 y al run:

`RUN_265ff1dcbdb94a6d95f49abc552ee758`

El checkpoint debe estar disponible en Google Drive bajo la estructura utilizada por el cuaderno:

`MyDrive/pln-sentimiento-estudiantil/transformer_ganador/<RUN_ID>/checkpoint-3036`

La evaluación final utiliza el commit:

`ddc3d2e51246e8450a0d000cd48af55bf101f878`

y puede ejecutarse en CPU con FP32. Esta etapa no realiza entrenamiento ni selección de hiperparámetros.

### Archivos de datos

Los archivos privados no están incluidos en el repositorio. Deben proporcionarse manualmente cuando el cuaderno los solicite:

- `train.csv`
- `validation.csv`
- `test.csv`
- `metadata.json`
- `baseline_C1_metadata.json`

Las verificaciones incorporadas en el proyecto comprueban tamaños, hashes y consistencia de las particiones antes de continuar.

### Reproducibilidad esperada

La reproducción del procedimiento está determinada por las particiones congeladas, la configuración del repositorio y los commits indicados. En entrenamiento con redes neuronales pueden existir pequeñas variaciones numéricas entre hardware o entornos, aun utilizando la misma semilla. La evaluación final del checkpoint congelado no modifica sus pesos.

## 1. Configuración del entorno

El entrenamiento de las configuraciones T1, T2 y T3 se realizó en Google Colab con una GPU Tesla T4 y precisión mixta FP16. Antes de iniciar los experimentos se verificaron PyTorch, CUDA y el dispositivo asignado.

La evaluación final sobre el conjunto de prueba se realizó posteriormente en CPU con precisión FP32 debido a la disponibilidad del entorno de ejecución. Este cambio afectó al tiempo de inferencia, pero no modificó los pesos, los hiperparámetros ni el checkpoint seleccionado.

In [ ]:
import json
import subprocess
import sys
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
from packaging.version import Version
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)

if not Version("2.6") <= Version(version("torch")) < Version("3"):
    raise RuntimeError("DETENER: PyTorch debe cumplir >=2.6,<3.")

if not torch.cuda.is_available() or torch.version.cuda is None:
    raise RuntimeError("DETENER: CUDA/GPU no está disponible.")

if torch.cuda.device_count() != 1:
    raise RuntimeError("DETENER: el protocolo oficial espera una sola GPU.")

probe = (torch.ones(1, device="cuda") * 2).item()
if probe != 2:
    raise RuntimeError("DETENER: el probe CUDA falló.")

free_bytes, total_bytes = torch.cuda.mem_get_info()

smi = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total,memory.free",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
    check=True,
)

before_versions = {}
for package in (
    "torch",
    "transformers",
    "tokenizers",
    "huggingface-hub",
    "accelerate",
    "numpy",
    "pandas",
    "scikit-learn",
    "joblib",
):
    try:
        before_versions[package] = version(package)
    except PackageNotFoundError:
        before_versions[package] = None

preflight = {
    "python": sys.version,
    "versions_before": before_versions,
    "torch_cuda_runtime": torch.version.cuda,
    "nvidia_smi": smi.stdout.strip(),
    "gpu_free_bytes_before": free_bytes,
    "gpu_total_bytes": total_bytes,
}

Path("/content/beto_colab_preflight.json").write_text(
    json.dumps(preflight, indent=2),
    encoding="utf-8",
)

print(smi.stdout.strip())
print("Preflight: OK")


Python: 3.13.15
PyTorch: 2.11.0+cpu
CUDA runtime: None


RuntimeError: DETENER: CUDA/GPU no está disponible.

## 2. Versiones del código y reproducibilidad

El entrenamiento oficial de T1, T2 y T3 se realizó con la versión del repositorio identificada por el commit:

`90afa082979bbe396f5a6a4906cadee9c351dd4f`

Con esa versión se entrenaron las tres configuraciones, se compararon sus resultados en validación y se seleccionó el checkpoint final.

La evaluación definitiva sobre el conjunto de prueba se ejecutó posteriormente con una versión más reciente del código:

`ddc3d2e51246e8450a0d000cd48af55bf101f878`

La segunda versión incorporó la infraestructura de evaluación final y soporte para inferencia en CPU. El cambio de versión no implicó reentrenamiento ni reselección: el modelo evaluado siguió siendo el checkpoint seleccionado previamente en validación.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO = Path("/content/pln-sentimiento-estudiantil")
REPO_URL = "https://github.com/diegoSpatinob/pln-sentimiento-estudiantil.git"
FINAL_COMMIT = "90afa082979bbe396f5a6a4906cadee9c351dd4f"

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

if not (REPO / ".git").exists():
    raise RuntimeError("DETENER: REPO existe pero no es un repositorio Git.")

dirty = subprocess.check_output(
    ["git", "status", "--porcelain"],
    cwd=REPO,
    text=True,
)

if dirty.strip():
    print(dirty)
    raise RuntimeError(
        "DETENER: hay cambios Git no ignorados; no se descartan automáticamente."
    )

subprocess.run(["git", "fetch", "origin"], cwd=REPO, check=True)
subprocess.run(
    ["git", "checkout", "--detach", FINAL_COMMIT],
    cwd=REPO,
    check=True,
)

os.chdir(REPO)

current_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

if current_commit != FINAL_COMMIT:
    raise RuntimeError(
        f"DETENER: commit activo {current_commit}; esperado {FINAL_COMMIT}"
    )

print("Repositorio:", REPO)
print("Commit activo:", current_commit)


## 3. Dependencias y configuración experimental

Se verificó la compatibilidad del entorno con las versiones utilizadas durante los experimentos. Para preservar la reproducibilidad, se mantuvieron las versiones validadas de las principales dependencias de Hugging Face.

El entrenamiento se ejecutó con GPU y FP16. La evaluación final se ejecutó con CPU y FP32, manteniendo sin cambios el modelo, el tokenizer, `max_length=128` y el checkpoint seleccionado.

In [ ]:
import json
import subprocess
import sys
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

# 3.1 — Comprobar configuración antes de importar Transformers.
config_path = REPO / "configs/transformer.json"
raw_config = json.loads(config_path.read_text(encoding="utf-8"))

expected_methodology = {
    "class_weight": None,
    "early_stopping": False,
    "oversampling": False,
    "effective_batch_size": 16,
    "physical_batch_size": 16,
    "gradient_accumulation_steps": 1,
    "precision": "fp16",
    "training_device": "cuda",
    "max_epochs": 3,
    "max_length": 128,
    "weight_decay": 0.01,
}

for field, expected in expected_methodology.items():
    actual = raw_config.get(field, "<FALTA>")
    if actual != expected:
        raise RuntimeError(
            f"DETENER: {field}={actual!r}; esperado={expected!r}."
        )

print("Configuración Fase 3A: OK")

# 3.2 — Preparar instalación HF.
required_hf = {
    "transformers": "4.57.1",
    "tokenizers": "0.22.2",
    "huggingface-hub": "0.36.2",
    "accelerate": "1.10.1",
}

to_install = [
    f"{name}=={expected}"
    for name, expected in required_hf.items()
    if (
        (lambda: (
            version(name)
            if True else None
        ))()
        if False else False
    )
]

# Construir lista sin forzar import de los paquetes.
to_install = []
for name, expected in required_hf.items():
    try:
        actual = version(name)
    except PackageNotFoundError:
        actual = None
    if actual != expected:
        to_install.append(f"{name}=={expected}")

print("Paquetes HF que requieren ajuste:", to_install or "ninguno")

if to_install:
    torch_constraint = Path("/content/beto_torch_constraint.txt")
    torch_constraint.write_text(
        f"torch=={version('torch')}\n",
        encoding="utf-8",
    )

    install_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade-strategy",
        "only-if-needed",
        "--constraint",
        str(torch_constraint),
        *to_install,
    ]

    # Primero dry-run.
    subprocess.run(
        [
            *install_command,
            "--dry-run",
            "--report",
            "/content/beto_install_plan.json",
        ],
        check=True,
    )

    plan = json.loads(
        Path("/content/beto_install_plan.json").read_text(encoding="utf-8")
    )

    if any(
        item["metadata"]["name"].lower() == "torch"
        for item in plan.get("install", [])
    ):
        raise RuntimeError("DETENER: pip propone modificar PyTorch.")

    subprocess.run(install_command, check=True)
    print("\nDependencias instaladas. REINICIA PYTHON antes de continuar.")
else:
    print("\nDependencias HF ya correctas. Puedes continuar sin reinstalar.")


### Reinicio del entorno

Cuando la instalación de dependencias modificó paquetes cargados en memoria, se reinició la sesión de Python y se reconstruyeron las variables necesarias antes de continuar.

## 4. Verificación del entorno después del reinicio

Tras reiniciar la sesión se comprobó nuevamente la disponibilidad del repositorio, las dependencias y el dispositivo de ejecución. Estas verificaciones permitieron asegurar que cada etapa se ejecutara con el entorno previsto.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json
from importlib.metadata import version

REPO = Path("/content/pln-sentimiento-estudiantil")
FINAL_COMMIT = "90afa082979bbe396f5a6a4906cadee9c351dd4f"

if not REPO.exists() or not (REPO / ".git").exists():
    raise RuntimeError(
        "DETENER: la VM se perdió. Ejecuta el notebook de nuevo desde la sección 1."
    )

subprocess.run(["git", "fetch", "origin"], cwd=REPO, check=True)
subprocess.run(
    ["git", "checkout", "--detach", FINAL_COMMIT],
    cwd=REPO,
    check=True,
)

os.chdir(REPO)

current_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

if current_commit != FINAL_COMMIT:
    raise RuntimeError("DETENER: no estamos en el commit oficial.")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

expected_hf = {
    "transformers": "4.57.1",
    "tokenizers": "0.22.2",
    "huggingface-hub": "0.36.2",
    "accelerate": "1.10.1",
}

for package, expected in expected_hf.items():
    actual = version(package)
    if actual != expected:
        raise RuntimeError(
            f"DETENER: {package}={actual}; esperado={expected}"
        )

from scripts.transformer import detect_runtime, load_config

runtime = detect_runtime("cuda")
config = load_config()

if runtime["device"] != "cuda" or len(runtime["gpu"]) != 1:
    raise RuntimeError("DETENER: se requiere una GPU CUDA.")

if not runtime["precision_support"]["fp16"]:
    raise RuntimeError("DETENER: la ejecución oficial requiere fp16.")

print("Commit:", current_commit)
print("GPU:", runtime["gpu"][0]["name"])
print("PyTorch:", runtime["torch"])
print("CUDA:", runtime["cuda_runtime"])
print("Transformers:", runtime["transformers"])
print("Accelerate:", version("accelerate"))
print("Precisión oficial: fp16")
print("Configuración cargada: OK")


## 5. Carga y verificación de las particiones de datos

Se cargan las particiones previamente definidas de entrenamiento, validación y prueba:

- `train.csv`
- `validation.csv`
- `test.csv`
- `metadata.json`

En esta etapa, el conjunto de prueba se utiliza únicamente para comprobar su integridad y correspondencia con la partición reservada. No se emplea para entrenar modelos ni para seleccionar hiperparámetros.

In [ ]:
import os
import tempfile
import hashlib

from scripts.fase0 import cargar_particiones

destination = REPO / "data/processed"
destination.mkdir(parents=True, exist_ok=True)

expected_files = {
    "train.csv",
    "validation.csv",
    "test.csv",
    "metadata.json",
}

present = {
    p.name
    for p in destination.iterdir()
    if p.is_file()
}

if expected_files - present:
    print("Faltan:", sorted(expected_files - present))
    from google.colab import files

    with tempfile.TemporaryDirectory(
        prefix="fase0_upload_",
        dir="/content",
    ) as staging:
        try:
            os.chdir(staging)
            uploaded = files.upload()
        finally:
            os.chdir(REPO)

        if set(uploaded) != expected_files:
            raise RuntimeError(
                "DETENER: selecciona exactamente los cuatro artefactos de Fase 0."
            )

        for name, content in uploaded.items():
            target = destination / name
            if target.exists() and target.read_bytes() != content:
                raise RuntimeError(
                    f"DETENER: {name} ya existe con bytes distintos."
                )

        for name, content in uploaded.items():
            target = destination / name
            if not target.exists():
                target.write_bytes(content)

partitions = cargar_particiones()

sizes = {
    name: len(frame)
    for name, frame in partitions.items()
}

expected_sizes = {
    "train": 16186,
    "validation": 3468,
    "test": 3469,
}

if sizes != expected_sizes:
    raise RuntimeError(f"DETENER: tamaños inesperados {sizes}")

data_hashes = {
    name: hashlib.sha256(
        (destination / name).read_bytes()
    ).hexdigest()
    for name in sorted(expected_files)
}

config_hash = hashlib.sha256(
    (REPO / "configs/transformer.json").read_bytes()
).hexdigest()

print("Particiones verificadas:", sizes)
print("Hashes: OK")

del partitions


## 6. Validación de la implementación

Antes del entrenamiento se ejecuta la suite de pruebas automatizadas del proyecto. Esta verificación comprueba la consistencia de la configuración, las particiones y las restricciones establecidas para los experimentos.

In [ ]:
import subprocess
import sys

tests = subprocess.run(
    [
        sys.executable,
        "-B",
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-v",
    ],
    cwd=REPO,
    capture_output=True,
    text=True,
    check=False,
)

print("RETURN CODE:", tests.returncode)
print("\n===== STDOUT =====")
print(tests.stdout)
print("\n===== STDERR =====")
print(tests.stderr)

if tests.returncode != 0:
    raise RuntimeError(
        "DETENER: la suite de tests falló. No entrenar."
    )

print("\nVALIDACIÓN PREVIA: OK")


## 7. Entrenamiento de las configuraciones T1, T2 y T3

Se realiza fine-tuning de tres configuraciones independientes de BETO. Cada experimento parte del modelo preentrenado original y utiliza la misma partición de entrenamiento y validación.

La única diferencia entre T1, T2 y T3 es la tasa de aprendizaje. Cada ejecución evalúa el modelo al final de cada época y conserva los checkpoints correspondientes para permitir la selección posterior con base en el rendimiento de validación.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

RESULT_PATHS = {}
RESULT_DATA = {}

def _completed_results(experiment):
    root = REPO / "results/transformer" / experiment
    if not root.exists():
        return set()
    return set(root.glob("*/validation_result.json"))

def _metric(metrics, name):
    if not isinstance(metrics, dict):
        return None
    for key in (name, f"eval_{name}"):
        if key in metrics:
            return metrics[key]
    return None

def show_result(experiment, path, data):
    print("\n" + "=" * 70)
    print(f"{experiment} TERMINÓ CORRECTAMENTE")
    print("=" * 70)
    print("Resultado:", path)
    print("Run ID:", data.get("run_id"))
    print("Learning rate:", data.get("learning_rate"))
    print("Test usado:", data.get("test_used"))

    epochs = data.get("validation_epochs", [])
    if epochs:
        print("\nValidation por época:")
        for row in epochs:
            metrics = row.get("metrics", row)
            epoch = row.get("epoch", metrics.get("epoch"))
            print(
                f"  época={epoch} | "
                f"F1-macro={_metric(metrics, 'f1_macro')} | "
                f"accuracy={_metric(metrics, 'accuracy')}"
            )

    selected = data.get("selected", {})
    print("\nSelección oficial:")
    print(json.dumps(selected, indent=2, ensure_ascii=False))

def run_experiment(experiment):
    if experiment not in {"T1", "T2", "T3"}:
        raise ValueError("Experimento inválido.")

    # La ejecución oficial exige árbol Git limpio.
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"],
        cwd=REPO,
        text=True,
    )
    if dirty.strip():
        raise RuntimeError(
            "DETENER: el repositorio no está limpio antes del experimento."
        )

    before = _completed_results(experiment)

    command = [
        sys.executable,
        "-u",
        "-m",
        "scripts.transformer",
        "--experiment",
        experiment,
    ]

    print("Ejecutando:", " ".join(command))

    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"

    completed = subprocess.run(
        command,
        cwd=REPO,
        env=env,
    )

    if completed.returncode != 0:
        raise RuntimeError(
            f"{experiment} falló. No continúes con el siguiente experimento."
        )

    after = _completed_results(experiment)
    new_results = after - before

    if len(new_results) != 1:
        raise RuntimeError(
            f"{experiment}: se esperaba exactamente un validation_result.json nuevo; "
            f"se encontraron {len(new_results)}."
        )

    path = new_results.pop()
    data = json.loads(path.read_text(encoding="utf-8"))

    if data.get("test_used") is not False:
        raise RuntimeError(
            f"{experiment}: el resultado no confirma test_used=false."
        )

    RESULT_PATHS[experiment] = path
    RESULT_DATA[experiment] = data
    show_result(experiment, path, data)

    return path


### 7.1 Experimento T1 — learning rate `1e-5`

Se entrena la primera configuración de BETO utilizando una tasa de aprendizaje de `1e-5`.

In [ ]:
T1_RESULT = run_experiment("T1")

### 7.2 Experimento T2 — learning rate `2e-5`

Se entrena la segunda configuración utilizando una tasa de aprendizaje de `2e-5`.

In [ ]:
T2_RESULT = run_experiment("T2")

### 7.3 Experimento T3 — learning rate `3e-5`

Se entrena la tercera configuración utilizando una tasa de aprendizaje de `3e-5`.

In [ ]:
T3_RESULT = run_experiment("T3")

## 8. Comparación en validación y selección del modelo

Los tres experimentos se comparan exclusivamente sobre el conjunto de validación.

La selección se realiza con el siguiente criterio:

1. mayor F1-macro;
2. en caso de empate a cuatro decimales, mayor accuracy;
3. si persiste el empate, menor tasa de aprendizaje;
4. finalmente, la época más temprana.

El conjunto de prueba no interviene en esta decisión.

In [ ]:
from datetime import datetime, timezone
import json
import subprocess
import sys

missing = [
    exp
    for exp in ("T1", "T2", "T3")
    if exp not in RESULT_PATHS
]

if missing:
    raise RuntimeError(
        f"DETENER: faltan resultados en esta sesión: {missing}"
    )

summary_output = (
    REPO
    / "results/transformer"
    / (
        "validation_summary_"
        + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + ".json"
    )
)

command = [
    sys.executable,
    "-m",
    "scripts.transformer",
    "--summarize",
    str(RESULT_PATHS["T1"]),
    str(RESULT_PATHS["T2"]),
    str(RESULT_PATHS["T3"]),
    "--summary-output",
    str(summary_output),
]

print("Ejecutando resumen oficial:")
print(" ".join(command))

completed = subprocess.run(command, cwd=REPO)

if completed.returncode != 0:
    raise RuntimeError(
        "DETENER: falló la comparación global de validation."
    )

if not summary_output.exists():
    raise RuntimeError(
        "DETENER: no se generó el JSON de resumen."
    )

validation_summary = json.loads(
    summary_output.read_text(encoding="utf-8")
)

print("\nRESUMEN GLOBAL DE VALIDATION")
print("=" * 70)
print(json.dumps(validation_summary, indent=2, ensure_ascii=False))
print("\nArchivo:", summary_output)


## 9. Identificación del modelo seleccionado

A partir de los resultados de validación se identifica la configuración con mejor desempeño y el checkpoint correspondiente a la época seleccionada.

In [ ]:
def _experiment_id(obj):
    if not isinstance(obj, dict):
        return None
    for key in ("experiment_id", "experiment", "id"):
        value = obj.get(key)
        if value in {"T1", "T2", "T3"}:
            return value
    return None

def detect_winner(summary):
    # Formas directas comunes.
    for key in (
        "selected_experiment",
        "winner",
        "global_winner",
        "selected",
    ):
        value = summary.get(key) if isinstance(summary, dict) else None

        if isinstance(value, str) and value in {"T1", "T2", "T3"}:
            return value

        exp = _experiment_id(value)
        if exp:
            return exp

    # Buscar una entrada marcada como ganadora.
    def walk(obj):
        if isinstance(obj, dict):
            exp = _experiment_id(obj)
            flags = (
                obj.get("winner"),
                obj.get("selected"),
                obj.get("is_selected"),
                obj.get("global_selected"),
            )
            if exp and any(flag is True for flag in flags):
                return exp
            for value in obj.values():
                found = walk(value)
                if found:
                    return found
        elif isinstance(obj, list):
            for value in obj:
                found = walk(value)
                if found:
                    return found
        return None

    return walk(summary)

WINNER = detect_winner(validation_summary)

if WINNER is None:
    print(
        "No pude detectar automáticamente el nombre del ganador, "
        "pero el resumen oficial se generó correctamente."
    )
    print("Revisa el JSON mostrado arriba antes de continuar con test final.")
else:
    winner_result = RESULT_DATA[WINNER]
    selected = winner_result.get("selected", {})

    print("GANADOR GLOBAL:", WINNER)
    print("Run ID:", winner_result.get("run_id"))
    print("Checkpoint seleccionado:", selected.get("checkpoint"))
    print("Época seleccionada:", selected.get("epoch"))
    print("Test usado:", winner_result.get("test_used"))


## 10. Conservación de resultados de validación

Se agrupan y descargan los resultados de validación de los tres experimentos junto con el resumen de selección. Estos artefactos permiten conservar evidencia de la comparación realizada sin incluir los pesos del modelo ni los datos privados.

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

archive = Path("/content/BETO_validation_resultados_OFICIALES.zip")

with zipfile.ZipFile(
    archive,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as bundle:
    for experiment in ("T1", "T2", "T3"):
        path = RESULT_PATHS[experiment]
        bundle.write(
            path,
            arcname=f"{experiment}_{path.parent.name}_validation_result.json",
        )

        manifest = path.parent / "run_manifest.json"
        if manifest.exists():
            bundle.write(
                manifest,
                arcname=f"{experiment}_{path.parent.name}_run_manifest.json",
            )

        epochs = path.parent / "validation_epochs.json"
        if epochs.exists():
            bundle.write(
                epochs,
                arcname=f"{experiment}_{path.parent.name}_validation_epochs.json",
            )

    bundle.write(
        summary_output,
        arcname=summary_output.name,
    )

print("ZIP generado:", archive)
files.download(str(archive))


## 11. Conservación del checkpoint seleccionado

El checkpoint del modelo seleccionado se copia a Google Drive para preservar sus pesos y los archivos necesarios para la evaluación final una vez finalizada la sesión de Colab.

In [ ]:
import shutil
from pathlib import Path

if WINNER is None:
    raise RuntimeError(
        "No hay WINNER detectado; no copiar checkpoints automáticamente."
    )

from google.colab import drive
drive.mount("/content/drive")

winner_data = RESULT_DATA[WINNER]
winner_run_id = winner_data.get("run_id")

if not winner_run_id:
    raise RuntimeError("El resultado ganador no contiene run_id.")

source_run = (
    REPO
    / "models/transformer"
    / WINNER
    / winner_run_id
)

if not source_run.exists():
    raise RuntimeError(
        f"No se encontró el directorio del run ganador: {source_run}"
    )

drive_root = (
    Path("/content/drive/MyDrive")
    / "pln-sentimiento-estudiantil"
    / "transformer_ganador"
)

drive_root.mkdir(parents=True, exist_ok=True)

destination_run = drive_root / source_run.name

if destination_run.exists():
    raise RuntimeError(
        f"DETENER: ya existe {destination_run}; no se sobrescribe."
    )

shutil.copytree(source_run, destination_run)
shutil.copy2(summary_output, drive_root / summary_output.name)

print("Checkpoint ganador guardado en:", destination_run)
print("Resumen guardado en:", drive_root / summary_output.name)


## 12. Cierre de la etapa de selección

Una vez entrenadas las tres configuraciones y seleccionado el mejor checkpoint mediante validación, la elección del modelo queda cerrada.

A partir de este punto no se modifican hiperparámetros ni se seleccionan nuevos checkpoints. La siguiente etapa corresponde únicamente a la evaluación final del modelo seleccionado sobre el conjunto de prueba reservado.

## Preparación para la evaluación final

A partir de esta sección ya no se realiza entrenamiento. El modelo T2 y su checkpoint fueron seleccionados previamente mediante validation.

Para reproducir únicamente esta etapa puede omitirse el bloque de entrenamiento anterior, siempre que el checkpoint seleccionado y los cinco artefactos privados requeridos estén disponibles.

La evaluación final utiliza una copia limpia del repositorio en el commit `ddc3d2e51246e8450a0d000cd48af55bf101f878` y ejecuta inferencia en CPU con FP32.

# 14. Evaluación final sobre el conjunto de prueba

El experimento T2 fue seleccionado previamente utilizando únicamente el conjunto de validación.

La configuración evaluada fue:

- Modelo: BETO cased
- Experimento: T2
- Learning rate: `2e-5`
- Checkpoint: `checkpoint-3036`
- Época seleccionada: 3

La selección quedó cerrada antes de utilizar el conjunto de prueba. Después de conocer los resultados de test no se realizó entrenamiento adicional, ajuste de hiperparámetros ni selección de otro checkpoint.

La inferencia final se ejecutó en CPU con precisión FP32. Aunque el entrenamiento oficial se realizó con GPU Tesla T4 y FP16, se reutilizaron exactamente los pesos del checkpoint previamente seleccionado en validation.

In [ ]:
from pathlib import Path
import subprocess
import os
import sys

REPO = Path("/content/pln-sentimiento-estudiantil-fase4-final")

REPO_URL = (
    "https:"
    + "//github.com/diegoSpatinob/"
    + "pln-sentimiento-estudiantil.git"
)

FINAL_COMMIT = "ddc3d2e51246e8450a0d000cd48af55bf101f878"

# Clonar solamente si no existe.
if not REPO.exists():
    clone = subprocess.run(
        ["git", "clone", REPO_URL, str(REPO)],
        capture_output=True,
        text=True,
    )

    print("git clone return code:", clone.returncode)

    if clone.stdout:
        print(clone.stdout)

    if clone.stderr:
        print(clone.stderr)

    if clone.returncode != 0:
        raise RuntimeError("DETENER: git clone falló.")

if not (REPO / ".git").exists():
    raise RuntimeError(
        f"DETENER: {REPO} existe pero no es un repositorio Git válido."
    )

# Asegurar origin correcto.
subprocess.run(
    ["git", "remote", "set-url", "origin", REPO_URL],
    cwd=REPO,
    check=True,
)

# Actualizar referencias.
fetch = subprocess.run(
    ["git", "fetch", "origin"],
    cwd=REPO,
    capture_output=True,
    text=True,
)

if fetch.returncode != 0:
    print(fetch.stdout)
    print(fetch.stderr)
    raise RuntimeError("DETENER: git fetch falló.")

# Checkout exacto al commit aprobado.
checkout = subprocess.run(
    ["git", "checkout", "--detach", FINAL_COMMIT],
    cwd=REPO,
    capture_output=True,
    text=True,
)

print("git checkout return code:", checkout.returncode)

if checkout.stdout:
    print(checkout.stdout)

if checkout.stderr:
    print(checkout.stderr)

if checkout.returncode != 0:
    raise RuntimeError("DETENER: checkout falló.")

os.chdir(REPO)

current = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

dirty = subprocess.check_output(
    ["git", "status", "--porcelain"],
    cwd=REPO,
    text=True,
).strip()

print("\nRepositorio:", REPO)
print("Commit activo:", current)
print("Git limpio:", not bool(dirty))

if current != FINAL_COMMIT:
    raise RuntimeError("DETENER: commit incorrecto.")

if dirty:
    raise RuntimeError(
        f"DETENER: working tree no limpio:\n{dirty}"
    )

print("\nCHECKOUT FINAL: OK")

git clone return code: 0
Cloning into '/content/pln-sentimiento-estudiantil-fase4-final'...

git checkout return code: 0
HEAD is now at ddc3d2e Merge pull request #8 from diegoSpatinob/fix/final-evaluation-cpu


Repositorio: /content/pln-sentimiento-estudiantil-fase4-final
Commit activo: ddc3d2e51246e8450a0d000cd48af55bf101f878
Git limpio: True

CHECKOUT FINAL: OK


In [ ]:
import subprocess
import sys
from importlib.metadata import version, PackageNotFoundError

EXPECTED_HF = {
    "transformers": "4.57.1",
    "tokenizers": "0.22.2",
    "huggingface-hub": "0.36.2",
    "accelerate": "1.10.1",
}

to_install = []

for package, expected in EXPECTED_HF.items():
    try:
        actual = version(package)
    except PackageNotFoundError:
        actual = None

    print(
        f"{package}: actual={actual} | esperado={expected}"
    )

    if actual != expected:
        to_install.append(
            f"{package}=={expected}"
        )

if not to_install:
    print("\nSTACK HF YA CORRECTO.")
    print("Puedes continuar directamente con la Celda 4.")

else:
    print("\nPaquetes a corregir:")

    for package in to_install:
        print(" -", package)

    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--force-reinstall",
            "--no-deps",
            *to_install,
        ],
        capture_output=True,
        text=True,
    )

    print("\nRETURN CODE:", result.returncode)

    if result.stdout:
        print("\nSTDOUT:")
        print(result.stdout)

    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            "DETENER: falló la instalación del stack Hugging Face."
        )

    print(
        "\nINSTALACIÓN CORRECTA."
        "\nAHORA REINICIA LA SESIÓN DE PYTHON."
    )

transformers: actual=4.57.1 | esperado=4.57.1
tokenizers: actual=0.22.2 | esperado=0.22.2
huggingface-hub: actual=0.36.2 | esperado=0.36.2
accelerate: actual=1.10.1 | esperado=1.10.1

STACK HF YA CORRECTO.
Puedes continuar directamente con la Celda 4.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO = Path("/content/pln-sentimiento-estudiantil-fase4-final")

FINAL_COMMIT = "ddc3d2e51246e8450a0d000cd48af55bf101f878"

if not REPO.exists():
    raise RuntimeError(
        "DETENER: la VM fue eliminada. "
        "Debes empezar nuevamente desde la Celda 2."
    )

if not (REPO / ".git").exists():
    raise RuntimeError(
        "DETENER: la carpeta ya no es un repositorio Git válido."
    )

os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

current = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

print("REPO:", REPO)
print("Commit:", current)

if current != FINAL_COMMIT:
    raise RuntimeError(
        "DETENER: el commit activo no es el aprobado."
    )

print("\nESTADO RECONSTRUIDO: OK")

REPO: /content/pln-sentimiento-estudiantil-fase4-final
Commit: ddc3d2e51246e8450a0d000cd48af55bf101f878

ESTADO RECONSTRUIDO: OK


In [ ]:
from importlib.metadata import version
import torch

EXPECTED_HF = {
    "transformers": "4.57.1",
    "tokenizers": "0.22.2",
    "huggingface-hub": "0.36.2",
    "accelerate": "1.10.1",
}

for package, expected in EXPECTED_HF.items():
    actual = version(package)

    print(f"{package}: {actual}")

    if actual != expected:
        raise RuntimeError(
            f"DETENER: {package}={actual}; esperado={expected}"
        )

print("\nPyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

print("\nENTORNO DE INFERENCIA: OK")

transformers: 4.57.1
tokenizers: 0.22.2
huggingface-hub: 0.36.2
accelerate: 1.10.1

PyTorch: 2.11.0+cpu
CUDA disponible: False

ENTORNO DE INFERENCIA: OK


In [ ]:
from google.colab import files
import os

os.chdir("/content")

uploaded = files.upload()

os.chdir(REPO)

print("\nArchivos recibidos:")

for name in uploaded:
    print(" -", name)

Saving metadata.json to metadata.json
Saving test.csv to test.csv
Saving train.csv to train.csv
Saving validation.csv to validation.csv
Saving baseline_C1_metadata.json to baseline_C1_metadata.json

Archivos recibidos:
 - metadata.json
 - test.csv
 - train.csv
 - validation.csv
 - baseline_C1_metadata.json


In [ ]:
from pathlib import Path
import re

DATA_DIR = REPO / "data/processed"
MODELS_DIR = REPO / "models"

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXPECTED_UPLOADS = {
    "train.csv",
    "validation.csv",
    "test.csv",
    "metadata.json",
    "baseline_C1_metadata.json",
}

def canonical_name(uploaded_name):

    for expected in EXPECTED_UPLOADS:

        stem, suffix = expected.rsplit(".", 1)

        if uploaded_name == expected:
            return expected

        pattern = (
            re.escape(stem)
            + r" \(\d+\)\."
            + re.escape(suffix)
        )

        if re.fullmatch(
            pattern,
            uploaded_name,
        ):
            return expected

    return None


normalized = {}

for uploaded_name, content in uploaded.items():

    canonical = canonical_name(uploaded_name)

    if canonical is None:
        continue

    if canonical in normalized:
        raise RuntimeError(
            f"DETENER: hay más de una versión de {canonical}."
        )

    normalized[canonical] = content


missing = (
    EXPECTED_UPLOADS
    - set(normalized)
)

if missing:
    raise RuntimeError(
        f"DETENER: faltan archivos: {sorted(missing)}"
    )


for name, content in normalized.items():

    if name == "baseline_C1_metadata.json":
        target = MODELS_DIR / name

    else:
        target = DATA_DIR / name

    target.write_bytes(content)

    print(
        f"{name} -> {target}"
    )

print("\nARCHIVOS PRIVADOS: OK")

metadata.json -> /content/pln-sentimiento-estudiantil-fase4-final/data/processed/metadata.json
test.csv -> /content/pln-sentimiento-estudiantil-fase4-final/data/processed/test.csv
train.csv -> /content/pln-sentimiento-estudiantil-fase4-final/data/processed/train.csv
validation.csv -> /content/pln-sentimiento-estudiantil-fase4-final/data/processed/validation.csv
baseline_C1_metadata.json -> /content/pln-sentimiento-estudiantil-fase4-final/models/baseline_C1_metadata.json

ARCHIVOS PRIVADOS: OK


In [ ]:
from scripts.fase0 import cargar_particiones

partitions = cargar_particiones()

sizes = {
    name: len(frame)
    for name, frame in partitions.items()
}

expected_sizes = {
    "train": 16186,
    "validation": 3468,
    "test": 3469,
}

print("Tamaños:", sizes)

if sizes != expected_sizes:
    raise RuntimeError(
        f"DETENER: tamaños inesperados: {sizes}"
    )

baseline_metadata = (
    REPO
    / "models"
    / "baseline_C1_metadata.json"
)

if not baseline_metadata.exists():
    raise RuntimeError(
        "DETENER: falta baseline_C1_metadata.json"
    )

print(
    "Metadata baseline presente:",
    baseline_metadata.exists(),
)

del partitions

print("\nARTEFACTOS CONGELADOS: OK")

Tamaños: {'train': 16186, 'validation': 3468, 'test': 3469}
Metadata baseline presente: True

ARTEFACTOS CONGELADOS: OK


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-B",
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-v",
    ],
    cwd=REPO,
    capture_output=True,
    text=True,
    check=False,
)

print(
    "RETURN CODE:",
    result.returncode,
)

print("\n===== STDOUT =====")
print(result.stdout)

print("\n===== STDERR =====")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "DETENER: los tests fallaron. "
        "NO abrir test."
    )

print("\nSUITE COMPLETA: OK")

RETURN CODE: 0

===== STDOUT =====
SMOKE_SAVE_RELOAD_DIAGNOSTICS {"purpose": "SMOKE_TEST_NO_OFICIAL", "save_reload_verified": true, "verification_precision": "fp32", "device": "cpu", "rtol": 0.0001, "atol": 0.0001, "state_dict_verified": true, "state_dict_tensor_count": 3, "different_state_keys": [], "unequal_state_tensors": [], "floating_state_dtypes": ["torch.float32"], "before_removing_amp_wrapper": {"max_abs_diff": 0.002239704132080078, "mean_abs_diff": 0.0019537508487701416, "finite": true, "original_dtype": "torch.bfloat16", "reloaded_dtype": "torch.float32", "comparison_dtype": "torch.float32", "allclose": false}, "equivalent_fp32_forwards": {"max_abs_diff": 0.0, "mean_abs_diff": 0.0, "finite": true, "original_dtype": "torch.float32", "reloaded_dtype": "torch.float32", "comparison_dtype": "torch.float32", "allclose": true}, "finite_losses": true}


===== STDERR =====
test_prediction_cli_requires_confirmation_and_rejects_development_splits (test_final_evaluation.CLITests.test_pre

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount(
    "/content/drive"
)

RUN_DIR = Path(
    "/content/drive/MyDrive/"
    "pln-sentimiento-estudiantil/"
    "transformer_ganador/"
    "RUN_265ff1dcbdb94a6d95f49abc552ee758"
)

CHECKPOINT = (
    RUN_DIR
    / "checkpoint-3036"
)

print("RUN:", RUN_DIR)
print("RUN existe:", RUN_DIR.exists())

print("\nCheckpoint:", CHECKPOINT)
print("Checkpoint existe:", CHECKPOINT.exists())

if not RUN_DIR.exists():
    raise RuntimeError(
        "DETENER: no se encontró el RUN ganador."
    )

if not CHECKPOINT.exists():
    raise RuntimeError(
        "DETENER: no se encontró checkpoint-3036."
    )

print("\nCHECKPOINT LOCALIZADO: OK")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUN: /content/drive/MyDrive/pln-sentimiento-estudiantil/transformer_ganador/RUN_265ff1dcbdb94a6d95f49abc552ee758
RUN existe: True

Checkpoint: /content/drive/MyDrive/pln-sentimiento-estudiantil/transformer_ganador/RUN_265ff1dcbdb94a6d95f49abc552ee758/checkpoint-3036
Checkpoint existe: True

CHECKPOINT LOCALIZADO: OK


In [ ]:
print("=== CONTENIDO DEL CHECKPOINT ===")

for path in sorted(
    CHECKPOINT.iterdir()
):
    print(" -", path.name)


config_file = (
    CHECKPOINT
    / "config.json"
)

safetensors_file = (
    CHECKPOINT
    / "model.safetensors"
)

pytorch_file = (
    CHECKPOINT
    / "pytorch_model.bin"
)


if not config_file.exists():
    raise RuntimeError(
        "DETENER: falta config.json."
    )

if not (
    safetensors_file.exists()
    or pytorch_file.exists()
):
    raise RuntimeError(
        "DETENER: faltan los pesos del modelo."
    )

print("\nCHECKPOINT GANADOR: OK")

=== CONTENIDO DEL CHECKPOINT ===
 - config.json
 - model.safetensors
 - optimizer.pt
 - rng_state.pth
 - scaler.pt
 - scheduler.pt
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer_config.json
 - trainer_state.json
 - training_args.bin
 - vocab.txt

CHECKPOINT GANADOR: OK


In [ ]:
FINAL_RESULTS = (
    REPO
    / "results"
    / "final"
)

print(
    "results/final existe:",
    FINAL_RESULTS.exists(),
)

if FINAL_RESULTS.exists():

    found = sorted(
        p.name
        for p in FINAL_RESULTS.iterdir()
    )

    print("Contenido:", found)

    if found:
        raise RuntimeError(
            "DETENER: hay artefactos de una evaluación anterior. "
            "NO borrar ni repetir. Revisar primero."
        )

else:
    print("Contenido: vacío")

print("\nSIN INTENTO FINAL PREVIO: OK")

results/final existe: False
Contenido: vacío

SIN INTENTO FINAL PREVIO: OK


In [ ]:
import subprocess

current = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

dirty = subprocess.check_output(
    ["git", "status", "--porcelain"],
    cwd=REPO,
    text=True,
).strip()

print("Commit:", current)
print("Git limpio:", not bool(dirty))

if current != FINAL_COMMIT:
    raise RuntimeError(
        "DETENER: commit incorrecto."
    )

if dirty:
    print(dirty)
    raise RuntimeError(
        "DETENER: working tree no limpio."
    )

print("\nPRE-FLIGHT FINAL: OK")

Commit: ddc3d2e51246e8450a0d000cd48af55bf101f878
Git limpio: True

PRE-FLIGHT FINAL: OK


## 15. Ejecución de la evaluación final

Se evaluó una única vez el checkpoint seleccionado de T2 sobre el conjunto de prueba reservado.

La evaluación utilizó los mismos pesos, tokenizer, longitud máxima y configuración seleccionados previamente mediante validation. Los resultados de test se utilizaron únicamente para estimar el rendimiento final del modelo y no para realizar una nueva selección.

In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    "-B",
    "-m",
    "scripts.final_evaluation",
    "transformer",
    "--checkpoint",
    str(CHECKPOINT),
    "--device",
    "cpu",
    "--confirm-final-test",
]

print(
    "INICIANDO EVALUACIÓN FINAL "
    "DEL TRANSFORMER T2 EN CPU..."
)

completed = subprocess.run(
    command,
    cwd=REPO,
)

print(
    "\nRETURN CODE:",
    completed.returncode,
)

if completed.returncode != 0:
    raise RuntimeError(
        "La evaluación final falló. "
        "NO vuelvas a ejecutar esta celda automáticamente. "
        "Primero debemos revisar la causa."
    )

print(
    "\nEVALUACIÓN FINAL TRANSFORMER: COMPLETADA"
)

INICIANDO EVALUACIÓN FINAL DEL TRANSFORMER T2 EN CPU...

RETURN CODE: 0

EVALUACIÓN FINAL TRANSFORMER: COMPLETADA


In [ ]:
import json
from pathlib import Path

TRANSFORMER_RESULT = (
    REPO
    / "results"
    / "final"
    / "transformer_T2_test.json"
)

if not TRANSFORMER_RESULT.exists():
    raise RuntimeError(
        "DETENER: no existe transformer_T2_test.json."
    )

transformer_test = json.loads(
    TRANSFORMER_RESULT.read_text(
        encoding="utf-8"
    )
)

print(
    json.dumps(
        transformer_test,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "confusion_matrix": {
    "columns": "predicho",
    "labels": [
      0,
      1
    ],
    "rows": "real",
    "values": [
      [
        1331,
        142
      ],
      [
        173,
        1823
      ]
    ]
  },
  "created_at_utc": "2026-10-04T07:40:51.294205+00:00",
  "data": {
    "metadata_sha256": "92a24524587855f76d0fbcb19a4e5513efe329f708d762c96d61ec879eeefcf4",
    "path": "data/processed/test.csv",
    "records": 3469,
    "sha256": "f1e7c5da7760aaf3ee0f89c3d86d1307db5a347cb1b543f551276b12e9c7851b"
  },
  "evaluation_git": {
    "commit": "ddc3d2e51246e8450a0d000cd48af55bf101f878",
    "diff_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "dirty": false,
    "transformer_script_sha256": "12f1c70b50c8035d403a2bf81f984fb25f3f6bc5d059822f43c6cdc168b056e2"
  },
  "evaluation_partition": "test",
  "evaluation_script_sha256": "40937c50f2a503342090382be2e5e8b1db397d1aa2cd1cee3b326dfe17097ebf",
  "label_mapping": {
    "Negativo": 0,
    "P

In [ ]:
import shutil
from pathlib import Path

BACKUP_DIR = Path(
    "/content/drive/MyDrive/"
    "pln-sentimiento-estudiantil/"
    "resultados_finales"
)

BACKUP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BACKUP_RESULT = (
    BACKUP_DIR
    / "transformer_T2_test.json"
)

if BACKUP_RESULT.exists():
    raise RuntimeError(
        "DETENER: transformer_T2_test.json "
        "ya existe en Drive. No se sobrescribe."
    )

shutil.copy2(
    TRANSFORMER_RESULT,
    BACKUP_RESULT,
)

print(
    "Resultado guardado en:"
)

print(
    BACKUP_RESULT
)

Resultado guardado en:
/content/drive/MyDrive/pln-sentimiento-estudiantil/resultados_finales/transformer_T2_test.json


In [ ]:
from google.colab import files

files.download(
    str(TRANSFORMER_RESULT)
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Nota de trazabilidad experimental

La secuencia experimental se mantuvo separada en dos momentos:

- **Entrenamiento y selección:** GPU Tesla T4, FP16, commit `90afa082979bbe396f5a6a4906cadee9c351dd4f`.
- **Evaluación final:** CPU, FP32, commit `ddc3d2e51246e8450a0d000cd48af55bf101f878`.

La segunda etapa reutilizó el checkpoint ya seleccionado y no modificó los parámetros aprendidos del modelo.

## 16. Resultados finales del Transformer

La evaluación final generó `transformer_T2_test.json`, que contiene las métricas obtenidas por T2 sobre el conjunto de prueba.

El modelo alcanzó un F1-macro de aproximadamente `0.9073` y una accuracy de aproximadamente `0.9092`. Estos resultados se utilizaron posteriormente para realizar una comparación descriptiva con el baseline clásico.

Una vez obtenidas las métricas de test, no se efectuaron nuevos entrenamientos, ajustes de hiperparámetros ni cambios de checkpoint.

## 17. Comparación con el baseline

El modelo Transformer seleccionado se comparó con el baseline clásico
TF-IDF + Regresión Logística utilizando exactamente el mismo conjunto de prueba
reservado de 3469 observaciones.

| Modelo | Accuracy | F1-macro | Precision macro | Recall macro |
|---|---:|---:|---:|---:|
| TF-IDF + Regresión Logística | 0.8694 | 0.8662 | 0.8667 | 0.8658 |
| BETO T2 | **0.9092** | **0.9073** | **0.9064** | **0.9085** |
| Diferencia T2 - baseline | **+0.0398** | **+0.0411** | **+0.0397** | **+0.0427** |

BETO T2 obtuvo un mejor resultado que el baseline en las cuatro métricas
globales consideradas. La mejora absoluta en F1-macro, métrica principal del
proyecto, fue de aproximadamente 0.0411.

Para la clase Negativo, el F1 aumentó de 0.8456 a 0.8942. Para la clase
Positivo, aumentó de 0.8869 a 0.9205.

La comparación sobre test se utilizó únicamente para describir el rendimiento
final de los modelos. No se realizó una nueva selección ni ajuste de
hiperparámetros a partir de estos resultados.

## 18. Conclusiones

Los experimentos permitieron evaluar tres configuraciones de BETO para la
clasificación binaria de sentimiento en comentarios de estudiantes
universitarios. La selección se realizó exclusivamente mediante el conjunto de
validación, utilizando F1-macro como métrica principal.

La configuración T2, con learning rate de 2e-5 y el checkpoint correspondiente
a la tercera época, presentó el mejor rendimiento en validación y fue
seleccionada antes de utilizar el conjunto de prueba.

En la evaluación final, BETO T2 alcanzó un F1-macro de 0.9073 y una accuracy de
0.9092, superando al baseline TF-IDF + Regresión Logística, cuyo F1-macro fue
0.8662. La mejora de aproximadamente 0.0411 en F1-macro también estuvo
acompañada por mejoras en precision macro, recall macro y en el F1 de ambas
clases.

Estos resultados indican que, para este conjunto de datos, el modelo
Transformer capturó mejor la información semántica de los comentarios que el
enfoque clásico basado en representación TF-IDF. La conclusión se limita al
conjunto de prueba reservado utilizado en este proyecto y no implica
generalización automática a otros dominios o poblaciones.